In [8]:
import numpy as np
from scipy.sparse import diags
from scipy.sparse.linalg import eigs

In [23]:
# Function to initialize P randomly
def initialize_P(n, k):
    P = np.random.rand(n, k)
    P /= np.sum(P, axis=1)[:, np.newaxis]
    return P

# Function to compute the regularized Laplacian (Lz)
def compute_regularized_laplacian(A, Z):
    return A + Z

# Non-backtracking matrix-based method to estimate number of communities
def estimate_num_communities(A):
    degrees = np.sum(A, axis=1)
    diag_degrees = diags(degrees)
    non_backtracking_matrix = np.block([[np.zeros_like(A), diag_degrees -np.eye(len(A))], [-np.eye(len(A)), A]])
    
    num_communities = 10  # Initial value
    while num_communities > 1:
        eigvals, _ = eigs(non_backtracking_matrix, k=num_communities, which='LM')
        real_eigvals = np.real(eigvals)
        if np.all(np.isreal(real_eigvals)):
            break
    return max(num_communities, 1)  # Ensure at least 2 communities

# Symmetric NMF update rule
def update_P(Lz, P, beta=0.5):
    numerator = np.dot(Lz, P)
    denominator = np.dot(np.dot(P, P.T), P)
    denominator[denominator == 0] = 1e-10  # Avoid division by zero
    P = P * ((1 - beta) + beta * numerator / denominator)
    return np.maximum(P, 1e-10)  # Ensure non-negativity

# CDNMF algorithm
def CDNMF(A, features, num_iterations=100, beta=0.5):
    n = A.shape[0]
    
    # Step 1: Estimate the number of communities (k) using spectral analysis
    k = estimate_num_communities(A)
    # k = 2  # For simplicity, we will use a fixed number of communities
    print(f"Estimated number of communities: {k}")
    
    # Step 2: Initialize P matrix randomly
    P = initialize_P(n, k)
    
    # Step 3: Initialize regularization matrix Z
    IPR = np.zeros(n)  # Store IPR for each node
    for i in range(n):
        IPR[i] = np.sum(features[i] ** 4)
    
    beta = 0.5  # Regularization parameter
    Z = np.diag(1 + beta * IPR)
    
    # Step 4: Compute regularized Laplacian Lz = A + Z
    Lz = compute_regularized_laplacian(A, Z)
    
    # Step 5: Iteratively update P using the update rule
    for iteration in range(num_iterations):
        P = update_P(Lz, P, beta)
        
        # Normalize P so that each row sums to 1
        P /= np.sum(P, axis=1)[:, np.newaxis]
    
    # Assign nodes to communities based on maximum membership value
    communities = np.argmax(P, axis=1)
    
    return communities, P


In [24]:
# Function to parse .cites and .content files and build the adjacency matrix
def parse_dataset(cites_file, content_file):
    # Parse the .content file
    features = {}
    node_id_to_index = {}
    num_nodes = 0
    with open(content_file, "r") as file:
        for line in file:
            parts = line.strip().split("\t")
            node_id = parts[0]
            node_id_to_index[node_id] = num_nodes
            feature_vector = np.array([float(x) for x in parts[1:-1]])
            features[num_nodes] = feature_vector
            num_nodes += 1

    # Parse the .cites file
    edges = []
    with open(cites_file, "r") as file:
        for line in file:
            parts = line.strip().split("\t")
            edges.append((parts[0], parts[1]))
    
    print(f"Number of nodes: {num_nodes}")
    print(f"Number of edges: {len(edges)}")
    print(f"Feature dimension: {len(features[0])}")
    
    # Build the adjacency matrix
    A = np.zeros((num_nodes, num_nodes))
    for edge in edges:
        if edge[0] not in features or edge[1] not in features:
            continue
        x = node_id_to_index[edge[0]]
        y = node_id_to_index[edge[1]]
        A[x, y] = 1
    
    return A, features

In [25]:
# Example usage
if __name__ == "__main__":
    # Load the dataset
    cites_file = "Datasets/citeseer-doc-classification/citeseer.cites"
    content_file = "Datasets/citeseer-doc-classification/citeseer.content"
    A, features = parse_dataset(cites_file, content_file)

    # Run the CDNMF algorithm
    communities, P = CDNMF(A, features, num_iterations=100, beta=0.5)
    
    # Output community assignment
    print("Community assignment:", communities)
    print("Community membership matrix P:\n", P)

Number of nodes: 3312
Number of edges: 4732
Feature dimension: 3703
Estimated number of communities: 10


KeyboardInterrupt: 